# 01b — Scrape ICWSM & JCDL Award Data

**ICWSM**: scraped from `icwsm.org/awards/`  
> ⚠️ The official ICWSM awards page only lists up to **2021**. Awards for 2022–2025 are missing from the source.

**JCDL**: scraped live from `jcdl.org/awards.php`.  
Page structure: one `<h2>` per award type, each followed by a `<table>` with columns `Year | Authors | Title`.  
Award types collected:
- Vannevar Bush Best Paper Award ← main award of interest  
- Best Student Paper Award  
- Best Short Paper Award  
- Best International Paper Award  
- Best Resource Paper Award  
- Best Poster Award  
- Best Demonstration Award  

All JCDL entries are then matched to OpenAlex via fuzzy title search.

**Output schema** (matches `huang_awards_cleaned.csv`):
```
year | conference | paper_title | paper_url | authors | award_type | award_year
```

Saved to: `../data/raw/icwsm_jcdl_awards_raw.csv`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, re
from rapidfuzz import fuzz

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
SIMILARITY_THRESHOLD = 85

## 1. Scrape ICWSM Awards

Page structure: `<h2>` = award type OR year, `<h3>` = paper title, `<h4>` = authors + original year

> ⚠️ Source only lists up to 2021 — gap is known and documented.

In [ ]:
def scrape_icwsm_awards():
    resp = requests.get("https://icwsm.org/awards/", headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None
    current_year = None
    pending_title = None

    AWARD_SECTIONS = {
        "test of time": "Test of Time",
        "best paper": "Best Paper",
        "outstanding paper": "Outstanding Paper",
        "honorable mention": "Honorable Mention",
    }

    for tag in soup.find_all(["h2", "h3", "h4"]):
        text = tag.get_text(separator=" ", strip=True)

        if tag.name == "h2":
            matched = False
            for kw, label in AWARD_SECTIONS.items():
                if re.search(kw, text, re.IGNORECASE):
                    current_award_type = label
                    matched = True
                    break
            if not matched:
                m = re.fullmatch(r"\s*(20\d{2})\s*", text)
                if m:
                    current_year = int(m.group(1))

        elif tag.name == "h3" and current_award_type:
            m = re.fullmatch(r"\s*(20\d{2})\s*", text)
            if m:
                current_year = int(m.group(1))
                pending_title = None
            else:
                pending_title = text

        elif tag.name == "h4" and pending_title and current_year and current_award_type:
            orig_year_m = re.search(r"ICWSM\s*(20\d{2})", text, re.IGNORECASE)
            pub_year = int(orig_year_m.group(1)) if orig_year_m else current_year
            authors = re.sub(r";?\s*ICWSM\s*20\d{2}.*$", "", text, flags=re.IGNORECASE).strip()
            records.append({
                "year": pub_year,
                "conference": "ICWSM",
                "paper_title": pending_title,
                "paper_url": "",
                "authors": authors,
                "award_type": current_award_type,
                "award_year": current_year,
            })
            pending_title = None

    return pd.DataFrame(records)


df_icwsm = scrape_icwsm_awards()
print(f"ICWSM records scraped: {len(df_icwsm)}")
print(f"Years covered: {sorted(df_icwsm['award_year'].unique())}")
df_icwsm

## 2. Scrape JCDL Awards

Page structure: multiple `<h2>` headings (one per award type), each followed by a `<table>` with columns `Year | Authors | Title`.  
We walk the DOM sibling-by-sibling: when we hit an `<h2>`, we normalise it to an award label;  
when we hit the next `<table>`, we parse all its rows under that label.

In [ ]:
AWARD_TYPE_MAP = {
    "vannevar": "Vannevar Bush Best Paper",
    "best paper": "Vannevar Bush Best Paper",
    "student": "Best Student Paper",
    "short": "Best Short Paper",
    "international": "Best International Paper",
    "resource": "Best Resource Paper",
    "poster": "Best Poster",
    "demonstration": "Best Demonstration",
    "demo": "Best Demonstration",
}

def normalize_award_type(h2_text):
    h = h2_text.lower().strip()
    for kw, label in AWARD_TYPE_MAP.items():
        if kw in h:
            return label
    return h2_text.strip()


def parse_jcdl_table(table, award_type):
    rows = table.find_all("tr")
    if not rows:
        return []

    # Detect column positions from header row
    header_cells = rows[0].find_all(["th", "td"])
    headers = [c.get_text(strip=True).lower() for c in header_cells]
    year_idx    = next((i for i, h in enumerate(headers) if "year"   in h), 0)
    authors_idx = next((i for i, h in enumerate(headers) if "author" in h), 1)
    title_idx   = next((i for i, h in enumerate(headers) if "title"  in h), 2)

    records = []
    for row in rows[1:]:
        cells = row.find_all(["td", "th"])
        if len(cells) < 2:
            continue
        year_text = cells[year_idx].get_text(strip=True) if year_idx < len(cells) else ""
        m = re.search(r"(\d{4})", year_text)
        if not m:
            continue
        year = int(m.group(1))

        title_cell = cells[title_idx] if title_idx < len(cells) else None
        title = title_cell.get_text(separator=" ", strip=True) if title_cell else ""
        paper_url = ""
        if title_cell:
            link = title_cell.find("a")
            if link and link.get("href"):
                paper_url = link["href"]

        authors_cell = cells[authors_idx] if authors_idx < len(cells) else None
        authors = authors_cell.get_text(separator=", ", strip=True) if authors_cell else ""

        if not title:
            continue

        records.append({
            "year": year,
            "conference": "JCDL",
            "paper_title": title,
            "paper_url": paper_url,
            "authors": authors,
            "award_type": award_type,
            "award_year": year,
        })
    return records


def scrape_jcdl_awards():
    resp = requests.get("https://jcdl.org/awards.php", headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    records = []
    current_award_type = None

    # Walk all top-level elements inside <body> (or main content div)
    for el in soup.find_all(["h2", "table"]):
        if el.name == "h2":
            current_award_type = normalize_award_type(el.get_text(separator=" ", strip=True))
            print(f"Found section: '{el.get_text(strip=True)}' → '{current_award_type}'")
        elif el.name == "table" and current_award_type:
            rows = parse_jcdl_table(el, current_award_type)
            records.extend(rows)
            current_award_type = None  # reset: each h2 owns exactly one table

    return pd.DataFrame(records)


df_jcdl_raw = scrape_jcdl_awards()
print(f"\nJCDL records scraped: {len(df_jcdl_raw)}")
print(df_jcdl_raw.groupby("award_type").size().reset_index(name="count").to_string(index=False))
df_jcdl_raw

## 3. Enrich JCDL with OpenAlex

Queries OpenAlex by title for each JCDL entry.  
Uses `rapidfuzz` to validate — if returned title similarity < 85, match is rejected.  
Adds: `openalex_id`, `openalex_title`, `cited_by_count`, enriched `authors`, `paper_url` (DOI).

In [ ]:
def enrich_with_openalex(query_title, mailto="thesis@example.com", threshold=SIMILARITY_THRESHOLD):
    params = {"search": query_title, "per_page": 3, "mailto": mailto}
    resp = requests.get("https://api.openalex.org/works", params=params, timeout=15)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        return None, "", ""

    best, best_score = None, 0
    for r in results:
        r_title = r.get("title") or r.get("display_name") or ""
        score = fuzz.token_sort_ratio(query_title.lower(), r_title.lower())
        if score > best_score:
            best, best_score = r, score

    if best_score < threshold:
        print(f"  SKIPPED (score={best_score:.1f}): best match → '{best.get('title', '')[:70]}'")
        return None, "", ""

    doi_url = f"https://doi.org/{best['doi'].split('doi.org/')[-1]}" if best.get("doi") else ""
    authors_str = "; ".join(
        a["author"]["display_name"] for a in best.get("authorships", []) if a.get("author")
    )
    return best, authors_str, doi_url


enriched_records = []
for _, row in df_jcdl_raw.iterrows():
    title = row["paper_title"]
    try:
        result, authors_str, doi_url = enrich_with_openalex(title)
        rec = row.to_dict()
        rec["openalex_id"]    = result["id"] if result else ""
        rec["openalex_title"] = result.get("title", "") if result else ""
        rec["cited_by_count"] = result["cited_by_count"] if result else None
        if result:
            rec["authors"]   = authors_str
            rec["paper_url"] = doi_url
        status = "OK" if result else "NO MATCH"
        print(f"{row['year']} [{row['award_type']}] {status}: {title[:65]}")
    except Exception as e:
        print(f"{row['year']} ERROR: {e}")
        rec = row.to_dict()
        rec["openalex_id"] = rec["openalex_title"] = ""
        rec["cited_by_count"] = None
    enriched_records.append(rec)
    time.sleep(0.3)

df_jcdl = pd.DataFrame(enriched_records)
matched = df_jcdl["openalex_id"].astype(bool).sum()
print(f"\nTotal JCDL: {len(df_jcdl)} | Matched: {matched} / {len(df_jcdl)}")
df_jcdl[["year", "award_type", "paper_title", "openalex_title", "cited_by_count"]]

## 4. Combine & Save

Merge ICWSM + JCDL into one clean CSV.  
Dedup key: `(paper_title, year, conference, award_type)` — a paper winning Best Paper AND Test of Time stays as two rows.

In [ ]:
COLS = ["year", "conference", "paper_title", "paper_url", "authors", "award_type", "award_year",
        "openalex_id", "openalex_title", "cited_by_count"]

for col in ["openalex_id", "openalex_title", "cited_by_count"]:
    if col not in df_icwsm.columns:
        df_icwsm[col] = ""

df_combined = pd.concat([df_icwsm[COLS], df_jcdl[COLS]], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=["paper_title", "year", "conference", "award_type"])
df_combined = df_combined.sort_values(["conference", "award_type", "year"]).reset_index(drop=True)

import os
os.makedirs("../data/raw", exist_ok=True)
out_path = "../data/raw/icwsm_jcdl_awards_raw.csv"
df_combined.to_csv(out_path, index=False)
print(f"Saved {len(df_combined)} records → {out_path}")

print("\nBreakdown:")
print(df_combined.groupby(["conference", "award_type"]).size().reset_index(name="count").to_string(index=False))